# Amazon Neptune Graph Session: a full-fledged graph in a smaller package

An Amazon Neptune Graph Session is a **full-fledged graph in a smaller package** — it runs the complete graph-algorithm catalog, exactly like a large provisioned graph. You get the same capabilities at a lower cost, with a one-line change to the requested size.

The full PaySim transaction graph has millions of edges, forcing you onto a large provisioned graph even when you only care about a small part of it. With an Amazon Neptune Graph Session, you project just a single day of activity (`step BETWEEN 0 AND 23`) — a focused slice that fits a smaller, cheaper graph while still running the full algorithm suite.

We'll:
1. Register the source data as an Athena table.
2. Create an 8 s-NCU Amazon Neptune Graph Session.
3. Project a single day's slice into it with a plain Athena SQL projection.
4. Run a graph algorithm on the result.


## Setup

Import the necessary libraries and set up logging.

In [ ]:
# Run this cell if kagglehub is not installed (e.g. on SageMaker)
import sys
!{sys.executable} -m pip install -q kagglehub dotenv

In [ ]:
# Check the Python version:
from sys import version_info
assert version_info >= (3, 11), "Python 3.11 or higher is required"

import logging
import os
import kagglehub
import boto3
from botocore.exceptions import ClientError
import dotenv
from pathlib import Path

from nx_neptune import empty_s3_bucket
from nx_neptune.session_manager import SessionManager
from nx_neptune.instance_management import execute_athena_query
from nx_neptune.utils.utils import get_stdout_logger, validate_and_get_env

In [ ]:
# Configure logging to see detailed information about the instance creation process
logger = get_stdout_logger(__name__, [
    'nx_neptune.instance_management',
    'nx_neptune.session_manager',
    __name__
])

## Configuration

Check for environment variables necessary for the notebook.

In [ ]:
dotenv.load_dotenv()
# Check for optional environment variables
env_vars = validate_and_get_env([
    'NETWORKX_S3_IMPORT_BUCKET_PATH',
    'NETWORKX_S3_TABLES_DATABASE',
    'NETWORKX_S3_TABLES_TABLENAME',
    'NETWORKX_STAGING_BUCKET',
    # Required to execute opencypher query outside of VPC.
    'NETWORKX_PUBLIC_CONNECTIVITY',
    'NETWORKX_DELETION_PROTECTION',
])

# Get environment variables
s3_location_import = os.getenv('NETWORKX_S3_IMPORT_BUCKET_PATH')
s3_staging_bucket = os.getenv('NETWORKX_STAGING_BUCKET')
s3_tables_catalog = os.getenv('NETWORKX_S3_TABLES_CATALOG')
s3_tables_database = os.getenv('NETWORKX_S3_TABLES_DATABASE')
s3_tables_tablename = os.getenv('NETWORKX_S3_TABLES_TABLENAME')
session_name = "nx-session-test"

## Data Setup

PaySim data is available from [kaggle](https://www.kaggle.com/code/kartik2112/fraud-detection-on-paysim-dataset/input?select=PS_20174392719_1491204439457_log.csv).

Data should be uploaded to an S3 bucket, and an athena table created for that bucket.

The PaySim dataset includes a simulated mobile money dataset, that involves transactions between client actors and banks. We can use this dataset to detect fraudulent activities in the simulated data.

In [ ]:
paysim_s3_bucket = s3_staging_bucket.replace('s3://', '').strip('/')
paysim_s3_bucket_path = 'datasets/paysim/'

# Download the latest version of paysim data
paysim_path = Path(kagglehub.dataset_download("ealaxi/paysim1"))
print("Path to paysim dataset files:", paysim_path)

# upload CSV to an S3 bucket if necessary
s3_client = boto3.client('s3')

for file_path in filter(Path.is_file, paysim_path.iterdir()):
  key = f"{paysim_s3_bucket_path}{file_path.name}"
  try:
      s3_client.head_object(Bucket=paysim_s3_bucket, Key=key)
      print(f"skip (exists): {key}")
  except ClientError:
      print(f"uploading: {key}")
      s3_client.upload_file(str(file_path), paysim_s3_bucket, key)


### Register the dataset to Athena

In [ ]:
# create CSV table from the uploaded data
s3_bucket_location = f"s3://{paysim_s3_bucket}/{paysim_s3_bucket_path}"

create_csv_table_stmt = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {s3_tables_tablename} (
    `step` int,
    `type` string,
    `amount` float,
    `nameOrig` string,
    `oldbalanceOrg` float,
    `newbalanceOrig` float,
    `nameDest` string,
    `oldbalanceDest` float,
    `newbalanceDest` float,
    `isFraud` int,
    `isFlaggedFraud` int
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe'
WITH SERDEPROPERTIES ('field.delim' = ',')
STORED AS INPUTFORMAT 'org.apache.hadoop.mapred.TextInputFormat'
OUTPUTFORMAT 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat'
LOCATION '{s3_bucket_location}'
TBLPROPERTIES ('classification' = 'csv', 'skip.header.line.count'='1');
"""

tasks = await execute_athena_query(
    create_csv_table_stmt,
    f"s3://{paysim_s3_bucket}/athena-results",
    database=s3_tables_database,
    catalog=s3_tables_catalog)

if tasks[0].done():
    print ("Create table completed")

## Obtain an Amazon Neptune Graph Session Instance

Provision a new Amazon Neptune Graph Session instance on demand, or retrieve an existing one.

In [ ]:
session = SessionManager.session(session_name)
graph_list = session.list_graphs()
print("The following graphs are available:")
for g in graph_list:
    print(g)

In [ ]:
session = SessionManager.session(session_name)
# In the case of full Paysim graph
# Use 16+ here for a standard provisioned graph instead.
graph = await session.get_or_create_graph(
  config={"computeConfiguration": {"graphSession": {"minMemory": 8}}})
print(f"Retrieved graph: {graph}")


## Project a single day's slice

We hand-write a plain Athena SQL projection into Neptune's load format (`~id`, `~label`, `~from`, `~to`). The `WHERE step BETWEEN 0 AND 23` predicate is what keeps this small — it selects only the first day of activity (~571K transactions and the ~819K accounts they touch), instead of all 6.3M rows. `import_from_table` runs the SQL, stages the CSV in S3, and loads it into the Amazon Neptune Graph Session.

In [ ]:
paysim_import_dir = f"{s3_location_import}/import_paysim_day0/"
# Clear any staged CSV from a previous run.
empty_s3_bucket(paysim_import_dir)

# Vertices: accounts that appear in a day-0 transaction.
CUSTOMERS = f'''
SELECT DISTINCT "~id", 'customer' AS "~label"
FROM (
   SELECT "nameOrig" AS "~id" FROM {s3_tables_tablename} WHERE "step" BETWEEN 0 AND 23
   UNION
   SELECT "nameDest" AS "~id" FROM {s3_tables_tablename} WHERE "step" BETWEEN 0 AND 23
)
'''


# Edges: the day-0 transactions themselves.
TRANSACTIONS = f'''
SELECT
  "nameOrig" AS "~from",
  "nameDest" AS "~to",
  "type"     AS "~label",
  "step"     AS "step:Int",
  "amount"   AS "amount:Float"
FROM {s3_tables_tablename}
WHERE "step" BETWEEN 0 AND 23
'''

await session.import_from_table(
    graph,
    paysim_import_dir,
    [CUSTOMERS, TRANSACTIONS],
    catalog=s3_tables_catalog,
    database=s3_tables_database,
)


## Run a graph algorithm

The projection is now a live Amazon Neptune Graph Session. As a quick demonstration that it imported correctly, we run Louvain community detection and annotate each of the graph vertices with the `community` property.

In [ ]:
# Sanity check: the graph has vertices and edges from the day-0 projection.
print(graph.execute_query("MATCH (n) RETURN count(n) AS nodes"))
print(graph.execute_query("MATCH ()-[r]->() RETURN count(r) AS edges"))

# Run Louvain and write each vertex's community into the "community" property.
# The mutate variant returns success: true when the graph was annotated.
louvain = graph.execute_query('''
    CALL neptune.algo.louvain.mutate({iterationTolerance:1e-07, writeProperty:"community"})
    YIELD success AS success
    RETURN success''')
print(f"Louvain succeeded: {louvain}")


# Clean up

Amazon Neptune Graph Sessions incur cost while they exist, so tear down the graph(s) created in this session once you're done. 
`destroy_all_graphs()` deletes every graph managed by this session.

In [ ]:
# destroy the Amazon Neptune Graph Sessions
await session.destroy_all_graphs()

## Takeaway

- **Full algorithm parity.** Louvain — and the rest of the catalog — runs on an Amazon Neptune Graph Session just as on a large one.
- **Size the graph to the projection.** A single-day (`step BETWEEN 0 AND 23`) slice is a small fraction of the data and fits an Amazon Neptune Graph Session comfortably; you pay for the focused sub-graph, not the full 6.3M-edge dataset.